# CCGA scratch notebook

Manual testing for the CCGA algebra. The `ccga` package is installed **editable** into this venv (`../.venv`, via `pyproject.toml`), so `import ccga` works and source edits are picked up live.

Select the **CCGA (.venv)** kernel.

In [ ]:
from ccga.algebra import (alg, e1, e2, eo, einf, eo1, eo2, eo3,
                          einf1, einf2, einf3, Iod, Iinfd, Io, Iinf, I, I_inv)
from ccga.point import point
from ccga.operations import join, meet, dual, undual, grades
from ccga.classify import classify
from ccga import print_null, format_null, to_null_basis
import ccga.objects as obj
from ccga.objects import make_point_ccga, make_round_point, make_point_pair

## Null-basis pretty-printer

`print_null` / `format_null` / `to_null_basis` express a multivector in the null
working basis `eo1 eo2 eo3 e1 e2 einf1 einf2 einf3` (any grade).

In [ ]:
print_null(point(2, 3))   # eo1 + eo2 + 2*e1 + 3*e2 + 2*einf1 + 4.5*einf2 + 6*einf3
print_null(Iod)           # eo1^eo3 - eo2^eo3
print_null(eo)
print_null(einf)

In [ ]:
# multiline=True: one blade per line, column-aligned, trailing sign
opns, _ = make_point_pair(point(1, 2), point(-1, 3))
print_null(opns, multiline=True)

In [ ]:
# IPNS / OPNS of an ellipse, shown in the null basis
opns, ipns = obj.make_ellipse(2, 1)
print('IPNS:', format_null(ipns))
print('grades(OPNS):', grades(opns))
print('classify(IPNS):', classify(ipns))
to_null_basis(ipns)

## CCGA point vs CGA round point

- `make_point_ccga(x, y, r=0)` — CCGA point (grade 1), radius optional.
- `make_round_point(x, y)` = `P ∧ Iinfd` — CGA round point (grade 3). Wedging
  with the infinity-gauge blade collapses the two CCGA quadratic coords into the
  single isotropic CGA term (x²+y²)/2 (§3.3 / §3.9).

In [ ]:
P = make_point_ccga(3, 4)
print('CCGA point  P, grade', grades(P), ':'); print_null(P)

R = make_round_point(3, 4)            # = P ^ Iinfd
print('\nCGA round point  P ^ Iinfd, grade', grades(R), ':')
print_null(R, multiline=True)
# last term: -12.5 * einf1^einf2^einf3   ==  -(3^2 + 4^2)/2

## CGA round-object family (via Iinfd)

Wedging with `Iinfd` recovers the whole CGA round hierarchy inside CCGA
(`ccga.cga`). Each CGA object of grade k lands at grade k+2:

| object | construction | grade |
|---|---|---|
| round point | `p ∧ Iinfd` | 3 |
| point pair | `p1 ∧ p2 ∧ Iinfd` | 4 |
| flat point | `p ∧ einf ∧ Iinfd` | 4 |
| circle | `p1 ∧ p2 ∧ p3 ∧ Iinfd` | 5 |
| line | `p1 ∧ p2 ∧ einf ∧ Iinfd` | 5 |

Reality is uniform: `reality(O) = sign((Iod | O)²)`.

In [ ]:
from ccga import cga
from ccga.objects import make_ideal_point

A, B, C = point(2, 0), point(-2, 0), point(0, 2)
family = {
    'round_point': cga.round_point(point(3, 4)),
    'sphere(r=2)': cga.round_point(make_point_ccga(1, 1, 2.0)),
    'sphere imag': cga.round_point(make_point_ccga(0, 0, 2.0, imaginary=True)),
    'point_pair':  cga.point_pair(A, B),
    'flat_point':  cga.flat_point(point(3, 4)),
    'circle':      cga.circle(A, B, C),
    'line':        cga.line(point(0, 0), point(2, 0)),
    'ideal round': cga.round_point(make_ideal_point(3, 4)),
}
for name, O in family.items():
    print(f'{name:12s}', cga.classify_cga(O))

In [ ]:
# the CGA circle through 3 points, shown in the null basis
print('circle, grade', grades(family['circle']), ':')
print_null(family['circle'], multiline=True)
# classify() now recognises the family and keeps the at-infinity labels:
from ccga.objects import make_line_at_infinity, make_conic_at_infinity
print('\nclassify(line_at_infinity) :', classify(make_line_at_infinity())['type'])
print('classify(conic_at_infinity):', classify(make_conic_at_infinity())['type'])

## Your scratch space